## Etapa 1: Algoritmo de Reconstrução de Lista (Parsing Estrutural)
Para contornar as falhas visuais do PDF (células mescladas sem linha divisória), extraímos o texto cru de cada linha e o transformamos em uma lista de strings. 

Em seguida, aplicamos um algoritmo de normalização com as seguintes regras de negócio:
1. **Ausência de Valor (`-`):** Se a lista retornar um traço, dividimos em Mínimo e Máximo e preenchemos ambos com `0.0`.
2. **Célula Mesclada (Valor Único):** Se houver apenas um resultado para a categoria, retornamos o mesmo valor para Mínimo e Máximo.
3. **Pares Válidos:** Dois números consecutivos formam um par (Mín e Máx) apenas se o primeiro for menor ou igual ao segundo.

In [2]:
import pdfplumber
import pandas as pd
import re
import os

def normalizar_lista_notas(lista_bruta):
    """
    Recebe a lista de notas crua e aplica regras para gerar exatamente 8 posições.
    """
    lista_final = []
    i = 0
    
    while i < len(lista_bruta):
        item = lista_bruta[i].strip()
        
        # Regra 1: Não retornou valor nenhum (-)
        if item == '-':
            lista_final.extend([0.0, 0.0])
            i += 1
        else:
            val_atual = float(item.replace(',', '.'))
            
            # Verifica se existe um próximo número na lista para formar par
            if i + 1 < len(lista_bruta) and lista_bruta[i+1].strip() != '-':
                val_prox = float(lista_bruta[i+1].replace(',', '.'))
                
                # Validação Matemática: Mínimo tem que ser <= Máximo
                if val_atual <= val_prox:
                    # É um par válido!
                    lista_final.extend([val_atual, val_prox])
                    i += 2
                    continue
            
            # Regra 2: Não se dividiu (valor único), repete para min e max
            lista_final.extend([val_atual, val_atual])
            i += 1
            
    # Caso faltem valores no final da linha, preenchemos com 0.0
    while len(lista_final) < 8:
        lista_final.append(0.0)
        
    return lista_final[:8]

def extrair_max_min_texto(caminho_pdf, ano):
    dados_finais = []
    
    # Regex: Captura o Curso e isola o bloco de notas/traços no final da linha
    regex_linha = re.compile(r"^(?P<curso>.*?)\s+(?P<notas>[\d,\s\-]+)$")
    
    with pdfplumber.open(caminho_pdf) as pdf:
        for pagina in pdf.pages:
            texto = pagina.extract_text()
            if not texto:
                continue
                
            for linha in texto.split('\n'):
                linha = linha.strip()
                
                # Ignora linhas de cabeçalho e títulos do documento
                if "CURSO" in linha or "MÍN" in linha or "MÁX" in linha or "VESTIBULAR" in linha or "PONTUAÇÃO" in linha:
                    continue
                    
                match = regex_linha.match(linha)
                if match:
                    curso_nome = match.group('curso').strip()
                    notas_str = match.group('notas').strip()
                    
                    # Quebra a string de notas em uma lista limpa (ex: ['53,75', '84,25', '45,00', '-'])
                    lista_bruta = [n for n in notas_str.split(' ') if n]
                    
                    # Aplica a sua regra de negócio
                    notas = normalizar_lista_notas(lista_bruta)
                    
                    # Utilizando Mínimo e Máximo como cabeçalhos definidos (4 categorias x 2 notas = 8 posições)
                    dados_curso = {
                        "Não Reservada": {"min": notas[0], "max": notas[1]},
                        "Rede Pública": {"min": notas[2], "max": notas[3]},
                        "Negros/Indígenas": {"min": notas[4], "max": notas[5]},
                        "Pessoas com Deficiência": {"min": notas[6], "max": notas[7]}
                    }
                    
                    for categoria, valores in dados_curso.items():
                        dados_finais.append({
                            'ano_vestibular': int(ano),
                            'curso': curso_nome,
                            'categoria_cota': categoria,
                            'nota_minima': valores['min'],
                            'nota_maxima': valores['max']
                        })
                        
    return pd.DataFrame(dados_finais)

ModuleNotFoundError: No module named 'pdfplumber'

## Etapa 2: Validação da Extração via Texto

In [12]:
arquivo_teste = "max_min/pontuacao-max-min-2018.pdf"
df_teste_notas = extrair_max_min_texto(arquivo_teste, "2018")

print("Tabela gerada através do algoritmo de lista e cabeçalhos:")
display(df_teste_notas.head(12))

Tabela gerada através do algoritmo de lista e cabeçalhos:


,ano_vestibular,curso,categoria_cota,nota_minima,nota_maxima
0,2018,Administração (RIO),Não Reservada,56.00,91.00
1,2018,Administração (RIO),Rede Pública,22.50,58.75
2,2018,Administração (RIO),Negros/Indígenas,21.50,56.50
3,2018,Administração (RIO),Pessoas com Deficiência,0.00,0.00
4,2018,Arqueologia (RIO),Não Reservada,53.75,84.25
5,2018,Arqueologia (RIO),Rede Pública,45.00,45.00
6,2018,Arqueologia (RIO),Negros/Indígenas,33.75,46.00
7,2018,Arqueologia (RIO),Pessoas com Deficiência,0.00,0.00
8,2018,Arquitetura e Urbanismo (PETRÓPOLIS),Não Reservada,73.75,87.00
9,2018,Arquitetura e Urbanismo (PETRÓPOLIS),Rede Pública,25.25,68.25


## Etapa 3: Ingestão em Lote (Batch Processing)

In [10]:
pasta_notas = "max_min"
lista_dfs_notas = []
arquivos_pdf_notas = [f for f in os.listdir(pasta_notas) if f.endswith('.pdf')]

print(f"Iniciando extração em lote de {len(arquivos_pdf_notas)} arquivos...\n")

for arquivo in arquivos_pdf_notas:
    caminho_completo = os.path.join(pasta_notas, arquivo)
    ano = arquivo.replace('.pdf', '')[-4:] 
    
    print(f"Processando ano {ano}...")
    df_ano = extrair_max_min_texto(caminho_completo, ano)
    lista_dfs_notas.append(df_ano)

df_notas_final = pd.concat(lista_dfs_notas, ignore_index=True)
print("\nProcessamento concluído com sucesso!")
df_notas_final.info()

Iniciando extração em lote de 9 arquivos...

Processando ano 2018...
Processando ano 2019...
Processando ano 2020...
Processando ano 2021...
Processando ano 2022...
Processando ano 2023...
Processando ano 2024...
Processando ano 2025...
Processando ano 2026...

Processamento concluído com sucesso!
<class 'pandas.DataFrame'>
RangeIndex: 2584 entries, 0 to 2583
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ano_vestibular  2584 non-null   int64  
 1   curso           2584 non-null   str    
 2   categoria_cota  2584 non-null   str    
 3   nota_minima     2584 non-null   float64
 4   nota_maxima     2584 non-null   float64
dtypes: float64(2), int64(1), str(2)
memory usage: 101.1 KB
